In [6]:
import pandas as pd

from morfeus import Sterimol
from morfeus import BuriedVolume
from morfeus import Pyramidalization
from morfeus import SASA
from morfeus import BiteAngle
from morfeus import SolidAngle
from morfeus import VisibleVolume

#import goodvibes.GoodVibes as gv
#import goodvibes.thermo as thermo
#import goodvibes.io as io
#import goodvibes.pes as pes

import dbstep.Dbstep as db

import numpy as np
import os
import json


In [ ]:
# Read XYZ file
with open('orca_example/optimization.xyz') as f:
    lines       = f.readlines()[2:]  # skip first two lines
data_list   = [line.split() for line in lines if line.strip()]
symbols     = [item[0] for item in data_list]
positions   = [[float(x) for x in item[1:4]] for item in data_list]

# Assume a list of atom indexes for which to calculate Sterimol parameters, dihedral angles, etc.
atom_indexes = [1, 2, 3, 4]  # Example indexes

class PropertiesXYZ:
    """Properties from XYZ file using Morfeus"""

    def __init__(self, path):
        """this class reads an xyz file from a given path and extracts properties using Morfeus
        Args:
            path (str): path to the folder containing optimization.xyz
        The class is initialized by reading the xyz file and storing the number of atoms, elements, and coordinates
        and reading a json file with results from DFT calculations.
        """
        if not path.endswith('/'):
            path = path + '/'
        self.path = path
        with open(path + 'optimization.xyz', 'r') as file:
            lines = file.readlines()
        self.n_atoms = int(lines[0].strip())
        self.elements = [line.split()[0] for line in lines[2:] if line.strip()]
        self.coordinates = [[float(x) for x in line.split()[1:4]] for line in lines[2:] if line.strip()]

        if 'xyz.json' in os.listdir(path):
            self.results = json.load(open(path + 'xyz.json'))
        else:
            print("No xyz.json file found in the specified path. Results dictionary will be empty.")
            self.results = {}

        # initialize lists for atoms, bonds, angles, dihedrals
        self.atom_list = None
        self.bond_list = None
        self.angle_list = None
        self.dihedral_list = None

        # default radius list for Vbur calculation
        self.radius_list = [2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]


    def define_atom_list(self, atom_list):
        self.atom_list = atom_list

    def define_bond_list(self, bond_list):
        self.bond_list = bond_list

    def define_angle_list(self, angle_list):
        self.angle_list = angle_list
    
    def define_dihedral_list(self, dihedral_list):
        self.dihedral_list = dihedral_list


    #-- Vbur calculation --#
    def get_Vbur(self):
        """
        Calculate buried volume for all atoms in atom_list at radius from 2.0 to 5.0 Å with a step of 0.5 Å
        The calculation is done using Morfeus.
        The atom_list must be defined before calling this function.
        The radius list is [2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0] by default but can be changed by modifying the radius parameter: self.radius_list
        Results are stored in the results dictionary with keys formatted as 'Vbur_atom_{at_idx}_radius_{radius}'.
        """
        if self.atom_list is None:
            print("Atom list is not defined. Please define it using define_atom_list().")
            return
        
        for at_idx in self.atom_list:
            for radius in self.radius_list:
                self.get_Vbur_at_radii(at_idx, radius)

            
    def get_Vbur_at_radii(self, at_idx, radius=3.5):
        """Calculate buried volume for a specific atom at a specific radius using Morfeus
        Args:
            at_idx (int): index of the atom (1-based)
            radius (float): radius in Å for the buried volume calculation
        """
        vbur = BuriedVolume(self.elements, self.coordinates, int(at_idx), include_hs=True, radius=radius) # calls Morfeus
        self.results['%Vbur_atom_' + str(self.elements[at_idx-1]) + str(at_idx) + '_radius_' + str(radius)] = vbur.fraction_buried_volume * 100

    #-- Bond Distances --#
    def get_bond_distances(self):
        """Calculate bond distances for all sets of two atoms in bond_list
        The bond_list must be defined before calling this function.
        Results are stored in the results dictionary with keys formatted as 'Bond_{atom1_symbol}{atom1}_{atom2_symbol}{atom2}'.
        """
        if self.bond_list is None:
            print("Bond list is not defined. Please define it using define_bond_list().")
            return
        
        for bond_atoms in self.bond_list:
            self.get_bond_distance(bond_atoms)

    def get_bond_distance(self, bond_list):
        """Calculate bond distance for a specific set of two atoms using Morfeus
        Args:
            bond_list (list): list of two atom indexes (1-based)
        """
        bond = Sterimol(self.elements, self.coordinates, bond_list[0], bond_list[1]) # calls Morfeus
        key = 'Bond_' + '_'.join([str(self.elements[i-1]) + str(i) for i in bond_list])
        self.results[key] = bond.distance


    #-- Angles --#
    def get_angles(self):
        """Calculate angles for all sets of three atoms in angle_list
        The angle_list must be defined before calling this function.
        Results are stored in the results dictionary with keys formatted as 'Angle_{atom1_symbol}{atom1}_{atom2_symbol}{atom2}_{atom3_symbol}{atom3}'.
        """
        if self.angle_list is None:
            print("Angle list is not defined. Please define it using define_angle_list().")
            return
        
        for angle_atoms in self.angle_list:
            self.get_angle(angle_atoms)


    def get_angle(self, angle_list):
        """Calculate angle for a specific set of three atoms using Morfeus
        Args:
            angle_list (list): list of three atom indexes (1-based)
        """
        angle = BiteAngle(self.elements, self.coordinates, angle_list[0], angle_list[1], angle_list[2]) # calls Morfeus
        key = 'Angle_' + '_'.join([str(self.elements[i-1]) + str(i) for i in angle_list])
        self.results[key] = angle.angle


    

In [27]:
P = PropertiesXYZ(path='orca_example/')
P.get_Vbur()
P.define_atom_list([1, 2])
P.get_Vbur()
P.results

No xyz.json file found in the specified path. Results dictionary will be empty.
Atom list is not defined. Please define it using define_atom_list().


{'%Vbur_atom_H1_radius_2.0': 33.703512396694215,
 '%Vbur_atom_H1_radius_2.5': 17.749706993098062,
 '%Vbur_atom_H1_radius_3.0': 10.255884982121573,
 '%Vbur_atom_H1_radius_3.5': 6.460818114051435,
 '%Vbur_atom_H1_radius_4.0': 4.325460865247346,
 '%Vbur_atom_H1_radius_4.5': 3.0403351858284124,
 '%Vbur_atom_H1_radius_5.0': 2.2145312352180633,
 '%Vbur_atom_H2_radius_2.0': 33.703512396694215,
 '%Vbur_atom_H2_radius_2.5': 17.749706993098062,
 '%Vbur_atom_H2_radius_3.0': 10.255884982121573,
 '%Vbur_atom_H2_radius_3.5': 6.460818114051435,
 '%Vbur_atom_H2_radius_4.0': 4.325460865247346,
 '%Vbur_atom_H2_radius_4.5': 3.0403351858284124,
 '%Vbur_atom_H2_radius_5.0': 2.2145312352180633}